# UAV Object Detection - Colab Training Notebook

This notebook clones the repo, pulls the dataset from DagsHub via DVC,
trains a YOLOv8 model on Colab's free T4 GPU, and pushes the trained
model weights back to DagsHub.

### Setup (one-time)
Before running, add your DagsHub token to **Colab Secrets**:
1. Click the **key icon** in the left sidebar of Colab.
2. Add a secret named `DAGSHUB_TOKEN` with your token as the value.
3. Toggle **"Notebook access"** ON.

## 1. Clone the GitHub Repository

In [ ]:
!git clone https://github.com/muhammadahmadamjad0/UAV-Object-Detection.git
%cd UAV-Object-Detection

## 2. Install Dependencies

In [ ]:
!pip install -r requirements.txt

## 3. Load Credentials Securely from Colab Secrets

This reads `DAGSHUB_TOKEN` from Colab's encrypted secret store.
Your token is **never** written into the notebook itself.

In [ ]:
from google.colab import userdata
import os

DAGSHUB_USERNAME = "muhammadahmadamjad0"
DAGSHUB_TOKEN = userdata.get("DAGSHUB_TOKEN")

# DVC credentials
os.environ["DVC_REMOTE_USER"] = DAGSHUB_USERNAME
os.environ["DVC_REMOTE_PASSWORD"] = DAGSHUB_TOKEN

# MLflow credentials
os.environ["MLFLOW_TRACKING_USERNAME"] = DAGSHUB_USERNAME
os.environ["MLFLOW_TRACKING_PASSWORD"] = DAGSHUB_TOKEN

print("Credentials loaded securely.")

## 4. Pull Dataset from DagsHub via DVC

In [ ]:
!dvc remote modify origin --local auth basic
!dvc remote modify origin --local user $DVC_REMOTE_USER
!dvc remote modify origin --local password $DVC_REMOTE_PASSWORD
!dvc pull

## 5. Train the YOLO Model

`dvc repro train_model` runs the training stage from `dvc.yaml`.
DVC checks if the dataset changed - if not, it skips regeneration
and goes straight to training. Metrics stream to DagsHub's MLflow.

In [ ]:
!dvc repro train_model

## 6. Push Trained Model Back to DagsHub

This uploads the `runs/` folder (trained weights) to DagsHub.
Any team member can then `dvc pull` to get the latest model.

In [ ]:
!dvc push
print("Model weights pushed to DagsHub successfully!")